# DB2 Warehouse Connectivity - Interaction

This notebook demonstrates how to connect to the Db2 warehouse instance, some basic functionality, as well as running queries in the database via python.

## Import Modules

In [22]:
import os
import ibm_db
import pandas as pd
import pyarrow.flight as flight
import itc_utils.flight_service as itcfs
from ibm_watson_studio_lib import access_project_or_space
from project_lib import Project
from pyspark.sql import SparkSession

#initiate spark session
sparkSession = SparkSession(spark).builder.getOrCreate()

## Interact with Db2 Warehouse

### Write to the database

In [27]:
# specify sql query to run
static_statement = """
CREATE TABLE "ERMINF_RAW"."ERMH_YEARLY_KWH_FOLIO"  (
		  "ACCT_COMPANY" CHAR(1 OCTETS) NOT NULL , 
		  "ACCT_FOLIO" CHAR(8 OCTETS) NOT NULL , 
		  "FY10" DOUBLE , 
		  "FY9" DOUBLE , 
		  "FY8" DOUBLE , 
		  "FY7" DOUBLE , 
		  "FY6" DOUBLE , 
		  "FY5" DOUBLE , 
		  "FY4" DOUBLE , 
		  "FY3" DOUBLE , 
		  "FY2" DOUBLE , 
		  "FY1" DOUBLE , 
		  "FY0" DOUBLE );
"""

In [29]:
# define connection to use and run query
nb_data_request = {
    'connection_name': """con-db2wh-1""",
    'interaction_properties': {
        'static_statement': static_statement,
        'write_mode': 'static_statement'
    },
}

flightClient = itcfs.get_flight_client()
flight_request = itcfs.get_data_request(nb_data_request=nb_data_request)
flight_request['context'] = 'target'

flight_cmd = itcfs.get_flight_cmd(data_request=flight_request)

action = flight.Action('setup_phase',flight_cmd.encode('utf-8'))
gen = flightClient.do_action(action)